# VAEと拡散モデル

世界モデルでは、速く扱える内部表現がほしい場面と、観測を丁寧に戻したい場面があります。VAE は軽い潜在表現を作る方向に寄せやすく、拡散モデルは反復的に観測品質を上げる方向に寄せやすい方法です。


## 圧縮してから戻すか、ノイズを削りながら戻すか

世界モデルの観測復元には、速く扱いやすい潜在空間が欲しい場面と、多少重くても見た目の品質を上げたい場面があります。VAE 系と拡散系は、その二つの要求に対して違う答えを出します。最初は、VAE を『いったん小さな作業メモへまとめてから戻す方法』、拡散を『壊れた見た目を少しずつ丁寧に直す方法』と捉えると入りやすくなります。


2 次元データを 1 次元に圧縮し、VAE 風の再構成と拡散風のデノイズを並べます。厳密な本実装ではありませんが、そのぶん『どこで速度を取り、どこで復元力を取るか』という設計の勘所が見えやすくなります。見るべきなのは勝敗ではなく、『先に軽い表現を作るか』『あとで見た目を磨き直すか』の役割分担です。


## 世界モデルの中で両者はどう住み分けるか

計画用の潜在状態には軽さと一貫性が欲しく、観測を見せる出口には細部を戻す力が欲しくなります。VAE 系は前者に、拡散系は後者に寄りやすいことが多く、その差を小さな例で確認します。言い換えると、『頭の中で先を試すための軽い内部表現』と『人に見せるための丁寧な復元』では、得意な作り方が違います。


## 見るべきポイント

`z` がどう圧縮されるか、PCA ベースの再構成で何が落ちるか、ノイズ除去反復でどこまで戻るかを比べながら読みます。ここで `z` は『圧縮後の短い作業メモ』、PCA は『まず一番大きな特徴だけ残す簡易な圧縮』だとして扱うと、後続の式とコードを追いやすくなります。


この数値は『どちらが絶対に優れているか』を決めるためではなく、同じデータに対して復元の作法がどう違うかを掴むためのものです。


## 近似実装で何を見ているか

VAE 側は PCA を encoder/decoder の近似に使い、拡散側は簡易スコア反復でデノイズを模します。encoder は見えているデータを内部メモへ写す係、decoder はそのメモから見た目へ戻す係です。細部は省いていますが、圧縮主体か高品質復元主体かという設計差は見えます。


## 読み方の軸

誤差が小さいかだけでなく、潜在空間を扱う軽さと、観測を戻す粘り強さのどちらを優先したい場面かを意識して見てください。『軽く回す内部表現がほしいのか』『重くてもきれいに戻したいのか』で使い分けを説明できる状態を目指します。


## 潜在を 1 次元へ落としてみる

まずは入力データの主な変動方向を 1 本に集めます。ここで失われる情報量が、あとで復元品質の差として現れます。つまり、最初の圧縮で何を捨てたかが、そのままあとで戻せない部分になります。


In [ ]:
import numpy as np
np.random.seed(3)

n = 300
theta = np.random.rand(n) * 2 * np.pi
x = np.stack([np.cos(theta), np.sin(theta)], axis=1) + 0.08 * np.random.randn(n, 2)


## 再構成とデノイズを並べてみる

次に、圧縮してすぐ戻す流れと、ノイズを少しずつ削って戻す流れを比較します。世界モデルの出口設計を考えるときの分岐点です。前者は『いったん小さくしてから戻す』流れ、後者は『壊れた状態から何度も手直しする』流れとして見てください。


In [ ]:
# VAE風: PCAを encoder/decoder の近似として利用
mu = x.mean(axis=0, keepdims=True)
xc = x - mu
u, s, vt = np.linalg.svd(xc, full_matrices=False)
pc1 = vt[0:1].T  # 2x1
z = xc @ pc1      # encoder
x_rec = z @ pc1.T + mu  # decoder
vae_mse = np.mean((x_rec - x) ** 2)

# Diffusion風: 潜在にノイズを加え、簡易スコア(ガウス事前)で反復デノイズ
z_noisy = z + 0.5 * np.random.randn(*z.shape)
z_den = z_noisy.copy()
prior_var = np.var(z_noisy) + 1e-8
for _ in range(30):
    score = -z_den / prior_var  # log N(0,var) の勾配
    z_den = z_den + 0.04 * score
x_den = z_den @ pc1.T + mu
diff_mse = np.mean((x_den - x) ** 2)

print('VAE-like reconstruction MSE       =', round(vae_mse, 6))
print('Diffusion-like latent denoise MSE =', round(diff_mse, 6))


## 圧縮の強さを変えて比べる

潜在を小さくすると速く扱いやすくなりますが、捨てた情報は戻しにくくなります。PCA の次元を 1 と 2 で変え、再構成誤差がどう変わるかを確認します。

In [ ]:
for latent_dim in [1, 2]:
    basis = vt[:latent_dim].T
    z_small = xc @ basis
    x_back = z_small @ basis.T + mu
    mse = np.mean((x_back - x) ** 2)
    print(f'latent_dim={latent_dim}: reconstruction MSE={mse:.6f}')

1 次元にすると内部表現は軽くなりますが、円周上の点を 1 本の軸へ押し込むため、戻したときに失う情報が増えます。2 次元なら元の形を保ちやすい一方、計画で持ち回る状態は大きくなります。世界モデルでは、軽さと復元力のどちらを優先するかを用途から決めます。

## 潜在で計画するときの計算量を比べる

潜在次元を小さくすると復元誤差は増えやすくなりますが、計画で持ち回る候補数と計算量は下がります。次のコードでは、候補行動列を評価するときに、潜在次元が増えるほど単純な演算量が増えることを確認します。


In [ ]:
candidate_plans = 128
horizon = 20
for latent_dim in [1, 2, 8, 32]:
    transition_ops = candidate_plans * horizon * latent_dim * latent_dim
    state_memory = candidate_plans * (horizon + 1) * latent_dim
    print(
        f'latent_dim={latent_dim:>2}: ops~{transition_ops:>7}, state_values={state_memory:>5}'
    )

復元品質だけを見れば大きい潜在が有利に見えます。しかし、計画で何百本もの候補を先読みするなら、潜在が大きいほど遷移計算と保存量が増えます。内部表現の軽さは、実時間で先を試すための性能要件です。


## デノイズの反復数を変えて比べる

拡散的な復元では、反復を増やすほど丁寧に直せますが、そのぶん計算時間が増えます。同じノイズから始めて、反復数を変えたときの誤差を比べます。

In [ ]:
for steps in [1, 5, 15, 30]:
    z_tmp = z_noisy.copy()
    for _ in range(steps):
        score = -z_tmp / prior_var
        z_tmp = z_tmp + 0.04 * score
    x_tmp = z_tmp @ pc1.T + mu
    mse = np.mean((x_tmp - x) ** 2)
    print(f'steps={steps:>2}: denoise MSE={mse:.6f}')

反復数は品質だけでなく遅さにも直結します。世界モデルの内部ロールアウトで毎回重い復元を回すと計画が遅くなります。だから、内部では軽い潜在を進め、必要な出口だけ丁寧に観測へ戻す、という分担が実装上も意味を持ちます。

## 復元品質と計画品質を別々に採点する

観測をきれいに戻せることと、計画に役立つことは同じではありません。次の例では、復元誤差、ロールアウト誤差、計画スコアの順位一致を別々に並べます。


In [ ]:
models = {
    'compact_vae': {'recon_mse': 0.060, 'rollout_mse': 0.035, 'rank_match': 0.82},
    'visual_diffusion': {'recon_mse': 0.018, 'rollout_mse': 0.070, 'rank_match': 0.58},
    'hybrid': {'recon_mse': 0.026, 'rollout_mse': 0.032, 'rank_match': 0.86},
}
for name, m in models.items():
    planning_score = m['rank_match'] - 2.0 * m['rollout_mse']
    print(name, 'planning_score=', round(planning_score, 3), 'recon_mse=', m['recon_mse'])

人に見せる映像なら復元誤差が重要ですが、制御ならロールアウト誤差や計画順位の一致が重要です。VAE と拡散を勝敗で覚えるより、内部で速く回す部分と出口で丁寧に戻す部分を分けて設計します。


要点は、VAE と拡散を対立で覚えることではありません。計画に使う軽い潜在と、見せるための高品質復元を同じモデルにどう分担させるか、その設計感覚を掴むことにあります。実務でも、『内部では軽く回し、出口だけ丁寧に作る』という分業が自然な場面は多く、その感覚を持てると、実装上の選択を誤りにくくなります。
